# Intro a Deep Learning: De Random Forest a Redes Neuronales

MNIST como campo de pruebas para entender el salto de ML clasico a deep learning:
perceptrones, MLPs, CNNs, y regularizacion.

**Dataset:** [MNIST](http://yann.lecun.com/exdb/mnist/) — 60k imagenes de digitos escritos a mano (28x28 px)
**Autor:** @aroaxinping
**Fecha:** Abril 2026

---
## 0. Configuracion del entorno

In [ ]:
import warnings
warnings.filterwarnings('ignore')

import time
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset

from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report

from pathlib import Path
import sys
sys.path.insert(0, str(Path('../src').resolve()))

# Estilo
plt.rcParams.update({
    'figure.facecolor': '#0f0f0f',
    'axes.facecolor':   '#1a1a1a',
    'axes.edgecolor':   '#333',
    'text.color':       '#e0e0e0',
    'axes.labelcolor':  '#e0e0e0',
    'xtick.color':      '#999',
    'ytick.color':      '#999',
    'grid.color':       '#2a2a2a',
    'grid.linestyle':   '--',
    'font.family':      'monospace',
    'axes.titlecolor':  '#ffffff',
    'axes.titlesize':   13,
    'axes.titleweight': 'bold',
})

ACCENT  = '#e85d04'   # naranja
ACCENT2 = '#6a9ad4'   # azul
ACCENT3 = '#2dc653'   # verde

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'PyTorch {torch.__version__} | Device: {device}')
print('Entorno listo.')

---
## 1. Datos (MNIST — 60k training images of handwritten digits 0-9)

| Dataset | Fuente | Registros |
|---|---|---|
| MNIST | [torchvision](https://pytorch.org/vision/stable/datasets.html#mnist) | 60k train + 10k test (real) / 12k (sintetico) |

> **Nota:** Si no tienes los datos descargados, el notebook genera un dataset sintetico con patrones simples por digito.

In [ ]:
# 1.1 Cargar datos
DATA_PATH = Path('../data/processed/mnist.npz')

if DATA_PATH.exists():
    data = np.load(DATA_PATH)
    X_train_raw, y_train = data['X_train'], data['y_train']
    X_test_raw, y_test = data['X_test'], data['y_test']
    es_sintetico = False
    print(f'[OK] Datos cargados: {DATA_PATH}')
else:
    print('[INFO] Datos no encontrados. Generando dataset sintetico...')
    print('[TIP]  Ejecuta: python src/fetch_dl_data.py')
    from fetch_dl_data import generate_synthetic_mnist
    X_train_raw, y_train, X_test_raw, y_test = generate_synthetic_mnist()
    es_sintetico = True

if es_sintetico:
    print('\nAVISO: Datos sinteticos. Patrones simples, no son digitos reales.')

print(f'\nTrain: {X_train_raw.shape} | Test: {X_test_raw.shape}')
print(f'Clases: {np.unique(y_train)}')
print(f'Rango pixeles: [{X_train_raw.min()}, {X_train_raw.max()}]')

In [ ]:
# 1.2 Muestra de imagenes
fig, axes = plt.subplots(3, 10, figsize=(14, 5))
for digit in range(10):
    idxs = np.where(y_train == digit)[0][:3]
    for row, idx in enumerate(idxs):
        ax = axes[row, digit]
        ax.imshow(X_train_raw[idx], cmap='gray')
        ax.set_xticks([])
        ax.set_yticks([])
        if row == 0:
            ax.set_title(str(digit), color=ACCENT, fontsize=14, fontweight='bold')

plt.suptitle('Muestra de MNIST — 3 ejemplos por digito', y=1.02, color='#fff', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

In [ ]:
# 1.3 Distribucion de clases
fig, axes = plt.subplots(1, 2, figsize=(14, 4))

for ax, labels, title in [
    (axes[0], y_train, 'Train'),
    (axes[1], y_test, 'Test'),
]:
    counts = np.bincount(labels, minlength=10)
    bars = ax.bar(range(10), counts, color=ACCENT, edgecolor='#333', alpha=0.85)
    ax.set(xlabel='Digito', ylabel='Frecuencia', title=f'Distribucion de clases — {title}')
    ax.set_xticks(range(10))
    for bar, count in zip(bars, counts):
        ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 50,
                str(count), ha='center', va='bottom', color='#999', fontsize=8)

plt.tight_layout()
plt.show()

In [ ]:
# 1.4 Distribucion de valores de pixel
fig, ax = plt.subplots(figsize=(10, 4))
sample_pixels = X_train_raw[:5000].flatten()
ax.hist(sample_pixels, bins=50, color=ACCENT, alpha=0.85, edgecolor='#333')
ax.axvline(sample_pixels.mean(), color=ACCENT2, ls='--', lw=1.5,
           label=f'media = {sample_pixels.mean():.1f}')
ax.set(xlabel='Valor de pixel (0-255)', ylabel='Frecuencia',
       title='Distribucion de valores de pixel (muestra de 5000 imagenes)')
ax.legend()
plt.tight_layout()
plt.show()

print(f'Pixeles = 0 (fondo negro): {(sample_pixels == 0).sum() / len(sample_pixels) * 100:.1f}%')
print(f'Pixeles > 0 (tinta):       {(sample_pixels > 0).sum() / len(sample_pixels) * 100:.1f}%')

---
## 2. Baseline: ML clasico

> **Pregunta:** Que accuracy da un Random Forest en imagenes aplanadas?

Antes de deep learning, veamos que tan bien funciona un modelo clasico.
Aplanamos cada imagen 28x28 en un vector de 784 features y entrenamos un Random Forest.

In [ ]:
# 2.1 Preparar datos para ML clasico
# Normalizar a [0, 1] y aplanar 28x28 -> 784
X_train_flat = X_train_raw.reshape(-1, 784).astype(np.float32) / 255.0
X_test_flat = X_test_raw.reshape(-1, 784).astype(np.float32) / 255.0

print(f'X_train_flat: {X_train_flat.shape}')
print(f'X_test_flat:  {X_test_flat.shape}')

In [ ]:
# 2.2 Random Forest
print('Entrenando Random Forest (100 arboles)...')
t0 = time.time()
rf = RandomForestClassifier(n_estimators=100, random_state=42, n_jobs=-1)
rf.fit(X_train_flat, y_train)
rf_train_time = time.time() - t0

y_pred_rf = rf.predict(X_test_flat)
acc_rf = accuracy_score(y_test, y_pred_rf)

print(f'\n--- Random Forest ---')
print(f'  Accuracy:  {acc_rf:.4f} ({acc_rf*100:.2f}%)')
print(f'  Tiempo:    {rf_train_time:.1f}s')
print(f'\n{classification_report(y_test, y_pred_rf)}')

In [ ]:
# 2.3 Confusion matrix — Random Forest
fig, ax = plt.subplots(figsize=(8, 7))
cm_rf = confusion_matrix(y_test, y_pred_rf)
sns.heatmap(cm_rf, annot=True, fmt='d', cmap='Oranges', ax=ax,
            xticklabels=range(10), yticklabels=range(10),
            linewidths=0.5, linecolor='#333')
ax.set(xlabel='Predicho', ylabel='Real', title=f'Confusion Matrix — Random Forest (acc={acc_rf:.3f})')
plt.tight_layout()
plt.show()

print('Este es nuestro baseline a superar con deep learning.')

---
## 3. Perceptron y redes neuronales

> **Pregunta:** Que es una neurona artificial y como se conectan en capas?

### La neurona artificial (perceptron)

Una neurona toma inputs, los multiplica por pesos, suma, y aplica una funcion de activacion:

```
  x1 --w1--\
  x2 --w2---[SUM + bias] --> f(z) --> output
  x3 --w3--/

  z = w1*x1 + w2*x2 + w3*x3 + bias
  output = f(z)
```

### Funciones de activacion

| Funcion | Formula | Uso |
|---|---|---|
| **Sigmoid** | 1 / (1 + e^(-z)) | Salida entre 0-1, clasificacion binaria |
| **ReLU** | max(0, z) | La mas comun en capas ocultas, simple y eficiente |
| **Softmax** | e^(zi) / sum(e^(zj)) | Capa final multiclase, da probabilidades |

### De una neurona a una red

```
  INPUT (784)     HIDDEN 1 (128)    HIDDEN 2 (64)     OUTPUT (10)
  [pixel 0]  -->  [neuron 0]  -->   [neuron 0]  -->   [P(digito=0)]
  [pixel 1]  -->  [neuron 1]  -->   [neuron 1]  -->   [P(digito=1)]
  [pixel 2]  -->  [neuron 2]  -->   [neuron 2]  -->   ...
  ...             ...               ...               [P(digito=9)]
  [pixel 783] ->  [neuron 127] ->   [neuron 63]
```

Cada neurona de una capa esta conectada a TODAS las neuronas de la siguiente (fully connected / dense).

In [ ]:
# 3.1 Visualizacion de funciones de activacion
fig, axes = plt.subplots(1, 3, figsize=(14, 4))
z = np.linspace(-5, 5, 200)

# Sigmoid
ax = axes[0]
sigmoid = 1 / (1 + np.exp(-z))
ax.plot(z, sigmoid, color=ACCENT, lw=2)
ax.axhline(0.5, color='#555', ls='--', lw=0.8)
ax.axvline(0, color='#555', ls='--', lw=0.8)
ax.set(title='Sigmoid', xlabel='z', ylabel='f(z)')
ax.set_ylim(-0.1, 1.1)

# ReLU
ax = axes[1]
relu = np.maximum(0, z)
ax.plot(z, relu, color=ACCENT2, lw=2)
ax.axvline(0, color='#555', ls='--', lw=0.8)
ax.set(title='ReLU', xlabel='z', ylabel='f(z)')

# Softmax (ejemplo con 3 clases)
ax = axes[2]
temps = [0.5, 1.0, 2.0]
z_sm = np.linspace(-3, 3, 200)
for t, c in zip(temps, [ACCENT, ACCENT2, ACCENT3]):
    softmax_val = np.exp(z_sm / t) / (np.exp(z_sm / t) + np.exp(0 / t) + np.exp(-1 / t))
    ax.plot(z_sm, softmax_val, color=c, lw=2, label=f'T={t}')
ax.set(title='Softmax (1 clase vs 2 fijas)', xlabel='z', ylabel='P(clase)')
ax.legend(fontsize=8)

plt.suptitle('Funciones de Activacion', y=1.02, color='#fff', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

---
## 4. Red neuronal simple (MLP)

> **Pregunta:** Una red de 2 capas supera al Random Forest?

Arquitectura: 784 (input) --> 128 (ReLU) --> 64 (ReLU) --> 10 (output)

In [ ]:
# 4.1 Preparar datos para PyTorch
# Nota: movemos TODO el dataset a GPU de una vez con .to(device).
# Esto funciona bien para MNIST (~180 MB), pero para datasets mas grandes
# (ImageNet, etc.) deberias mantener los datos en CPU y mover solo cada batch
# a GPU dentro del bucle de entrenamiento para no agotar la VRAM.
X_train_tensor = torch.FloatTensor(X_train_flat).to(device)
y_train_tensor = torch.LongTensor(y_train).to(device)
X_test_tensor = torch.FloatTensor(X_test_flat).to(device)
y_test_tensor = torch.LongTensor(y_test).to(device)

train_dataset = TensorDataset(X_train_tensor, y_train_tensor)
train_loader = DataLoader(train_dataset, batch_size=128, shuffle=True)

print(f'Train tensors: X={X_train_tensor.shape}, y={y_train_tensor.shape}')
print(f'Batches por epoch: {len(train_loader)}')

In [ ]:
# 4.2 Definir MLP
class MLP(nn.Module):
    def __init__(self):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(784, 128),
            nn.ReLU(),
            nn.Linear(128, 64),
            nn.ReLU(),
            nn.Linear(64, 10),
        )

    def forward(self, x):
        return self.net(x)

mlp = MLP().to(device)

# Contar parametros
n_params_mlp = sum(p.numel() for p in mlp.parameters())
print(f'Arquitectura MLP:')
print(mlp)
print(f'\nParametros totales: {n_params_mlp:,}')

In [ ]:
# 4.3 Funcion de entrenamiento (reutilizable)
def train_model(model, train_loader, X_test, y_test, epochs=20, lr=0.001):
    """Entrena un modelo y devuelve historial de metricas."""
    criterion = nn.CrossEntropyLoss()
    optimizer = optim.Adam(model.parameters(), lr=lr)

    history = {'train_loss': [], 'train_acc': [], 'test_loss': [], 'test_acc': []}

    t0 = time.time()
    for epoch in range(epochs):
        model.train()
        running_loss = 0.0
        correct = 0
        total = 0

        for X_batch, y_batch in train_loader:
            optimizer.zero_grad()
            outputs = model(X_batch)
            loss = criterion(outputs, y_batch)
            loss.backward()
            optimizer.step()

            running_loss += loss.item() * X_batch.size(0)
            _, predicted = outputs.max(1)
            total += y_batch.size(0)
            correct += predicted.eq(y_batch).sum().item()

        train_loss = running_loss / total
        train_acc = correct / total

        # Evaluacion en test
        model.eval()
        with torch.no_grad():
            test_outputs = model(X_test)
            test_loss = criterion(test_outputs, y_test).item()
            _, test_pred = test_outputs.max(1)
            test_acc = test_pred.eq(y_test).sum().item() / len(y_test)

        history['train_loss'].append(train_loss)
        history['train_acc'].append(train_acc)
        history['test_loss'].append(test_loss)
        history['test_acc'].append(test_acc)

        if (epoch + 1) % 5 == 0 or epoch == 0:
            print(f'  Epoch {epoch+1:2d}/{epochs} | '
                  f'Train Loss: {train_loss:.4f} Acc: {train_acc:.4f} | '
                  f'Test Loss: {test_loss:.4f} Acc: {test_acc:.4f}')

    elapsed = time.time() - t0
    print(f'\n  Tiempo total: {elapsed:.1f}s')
    history['time'] = elapsed
    return history

In [ ]:
# 4.4 Entrenar MLP
print('--- Entrenando MLP ---')
mlp = MLP().to(device)
history_mlp = train_model(mlp, train_loader, X_test_tensor, y_test_tensor, epochs=20)

In [ ]:
# 4.5 Curvas de entrenamiento MLP
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

ax = axes[0]
ax.plot(history_mlp['train_loss'], color=ACCENT, lw=2, label='Train')
ax.plot(history_mlp['test_loss'], color=ACCENT2, lw=2, label='Test')
ax.set(xlabel='Epoch', ylabel='Loss', title='MLP — Loss por Epoch')
ax.legend()

ax = axes[1]
ax.plot(history_mlp['train_acc'], color=ACCENT, lw=2, label='Train')
ax.plot(history_mlp['test_acc'], color=ACCENT2, lw=2, label='Test')
ax.axhline(acc_rf, color=ACCENT3, ls='--', lw=1.5, label=f'RF baseline ({acc_rf:.3f})')
ax.set(xlabel='Epoch', ylabel='Accuracy', title='MLP — Accuracy por Epoch')
ax.legend()

plt.tight_layout()
plt.show()

In [ ]:
# 4.6 Evaluacion MLP — Confusion matrix
mlp.eval()
with torch.no_grad():
    y_pred_mlp = mlp(X_test_tensor).argmax(1).cpu().numpy()

acc_mlp = accuracy_score(y_test, y_pred_mlp)
cm_mlp = confusion_matrix(y_test, y_pred_mlp)

fig, axes = plt.subplots(1, 2, figsize=(16, 7))

for ax, cm, title in [
    (axes[0], cm_rf, f'Random Forest (acc={acc_rf:.3f})'),
    (axes[1], cm_mlp, f'MLP (acc={acc_mlp:.3f})'),
]:
    sns.heatmap(cm, annot=True, fmt='d', cmap='Oranges', ax=ax,
                xticklabels=range(10), yticklabels=range(10),
                linewidths=0.5, linecolor='#333')
    ax.set(xlabel='Predicho', ylabel='Real', title=title)

plt.suptitle('Confusion Matrix — RF vs MLP', y=1.02, color='#fff', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

print(f'Random Forest: {acc_rf:.4f}')
print(f'MLP:           {acc_mlp:.4f}')
print(f'Diferencia:    {(acc_mlp - acc_rf)*100:+.2f} puntos porcentuales')

---
## 5. Red convolucional (CNN)

> **Pregunta:** Una CNN que "ve" la estructura 2D de la imagen es mejor que el MLP?

La CNN no aplana la imagen. En vez de eso, aplica filtros (kernels) que detectan patrones locales:
bordes, curvas, esquinas. Esto respeta la estructura espacial de la imagen.

Arquitectura:
```
Input (1x28x28)
  --> Conv2d(1, 32, 3, padding=1) + ReLU + MaxPool(2)   --> (32x14x14)
  --> Conv2d(32, 64, 3, padding=1) + ReLU + MaxPool(2)   --> (64x7x7)
  --> Flatten                                             --> (3136)
  --> Linear(3136, 128) + ReLU                            --> (128)
  --> Linear(128, 10)                                     --> (10)
```

In [ ]:
# 5.1 Preparar datos 2D para CNN (necesita canal: batch x 1 x 28 x 28)
# Nota: igual que antes, cargamos todo en GPU porque MNIST cabe en memoria.
# Para datasets grandes, mantendrias los tensores en CPU y usarias
# DataLoader con pin_memory=True + batch.to(device) en el loop de entrenamiento.
X_train_2d = torch.FloatTensor(X_train_raw.astype(np.float32) / 255.0).unsqueeze(1).to(device)
X_test_2d = torch.FloatTensor(X_test_raw.astype(np.float32) / 255.0).unsqueeze(1).to(device)

train_dataset_2d = TensorDataset(X_train_2d, y_train_tensor)
train_loader_2d = DataLoader(train_dataset_2d, batch_size=128, shuffle=True)

print(f'Shape para CNN: {X_train_2d.shape}  (batch, canales, alto, ancho)')

In [ ]:
# 5.2 Definir CNN
class CNN(nn.Module):
    def __init__(self, dropout_rate=0.0):
        super().__init__()
        self.features = nn.Sequential(
            nn.Conv2d(1, 32, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2),
            nn.Conv2d(32, 64, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2),
        )
        self.classifier = nn.Sequential(
            nn.Flatten(),
            nn.Linear(64 * 7 * 7, 128),
            nn.ReLU(),
            nn.Dropout(dropout_rate),
            nn.Linear(128, 10),
        )

    def forward(self, x):
        x = self.features(x)
        x = self.classifier(x)
        return x

cnn = CNN().to(device)
n_params_cnn = sum(p.numel() for p in cnn.parameters())
print(f'Arquitectura CNN:')
print(cnn)
print(f'\nParametros totales: {n_params_cnn:,}')

In [ ]:
# 5.3 Entrenar CNN
print('--- Entrenando CNN ---')
cnn = CNN().to(device)
history_cnn = train_model(cnn, train_loader_2d, X_test_2d, y_test_tensor, epochs=20)

In [ ]:
# 5.4 Curvas de entrenamiento CNN
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

ax = axes[0]
ax.plot(history_cnn['train_loss'], color=ACCENT, lw=2, label='Train')
ax.plot(history_cnn['test_loss'], color=ACCENT2, lw=2, label='Test')
ax.set(xlabel='Epoch', ylabel='Loss', title='CNN — Loss por Epoch')
ax.legend()

ax = axes[1]
ax.plot(history_cnn['train_acc'], color=ACCENT, lw=2, label='Train')
ax.plot(history_cnn['test_acc'], color=ACCENT2, lw=2, label='Test')
ax.axhline(acc_rf, color=ACCENT3, ls='--', lw=1.5, label=f'RF baseline ({acc_rf:.3f})')
ax.axhline(acc_mlp, color='#999', ls=':', lw=1.5, label=f'MLP ({acc_mlp:.3f})')
ax.set(xlabel='Epoch', ylabel='Accuracy', title='CNN — Accuracy por Epoch')
ax.legend()

plt.tight_layout()
plt.show()

In [ ]:
# 5.5 Evaluacion CNN + confusion matrix
cnn.eval()
with torch.no_grad():
    y_pred_cnn = cnn(X_test_2d).argmax(1).cpu().numpy()

acc_cnn = accuracy_score(y_test, y_pred_cnn)
cm_cnn = confusion_matrix(y_test, y_pred_cnn)

fig, ax = plt.subplots(figsize=(8, 7))
sns.heatmap(cm_cnn, annot=True, fmt='d', cmap='Oranges', ax=ax,
            xticklabels=range(10), yticklabels=range(10),
            linewidths=0.5, linecolor='#333')
ax.set(xlabel='Predicho', ylabel='Real', title=f'Confusion Matrix — CNN (acc={acc_cnn:.3f})')
plt.tight_layout()
plt.show()

print(f'CNN accuracy: {acc_cnn:.4f}')

In [ ]:
# 5.6 Ejemplos mal clasificados por la CNN
misclassified = np.where(y_pred_cnn != y_test)[0]
n_show = min(20, len(misclassified))

if n_show > 0:
    fig, axes = plt.subplots(2, 10, figsize=(14, 4))
    for i, idx in enumerate(misclassified[:n_show]):
        ax = axes[i // 10, i % 10]
        ax.imshow(X_test_raw[idx], cmap='gray')
        ax.set_title(f'{y_test[idx]}->{y_pred_cnn[idx]}', color='#ff4444', fontsize=9)
        ax.set_xticks([])
        ax.set_yticks([])
    # Ocultar ejes vacios
    for i in range(n_show, 20):
        axes[i // 10, i % 10].axis('off')

    plt.suptitle('Errores de la CNN (real -> predicho)', y=1.02, color='#fff', fontsize=14, fontweight='bold')
    plt.tight_layout()
    plt.show()
    print(f'Total errores: {len(misclassified)} de {len(y_test)} ({len(misclassified)/len(y_test)*100:.2f}%)')
else:
    print('La CNN clasifico todo correctamente.')

---
## 6. Overfitting y regularizacion

> **Pregunta:** Como evitar que la red memorice en vez de aprender?

Cuando el train accuracy sube pero el test accuracy se estanca o baja, la red esta memorizando
los datos de entrenamiento en vez de aprender patrones generalizables. Soluciones:

- **Dropout:** Apagar neuronas al azar durante el entrenamiento (fuerza redundancia)
- **Early stopping:** Parar de entrenar cuando el test loss deja de bajar
- **Data augmentation:** Generar variaciones de los datos (rotaciones, zoom...)

In [ ]:
# 6.1 Entrenar CNN sin dropout (mas epochs para provocar overfitting)
print('--- CNN sin Dropout (30 epochs para ver overfitting) ---')
cnn_overfit = CNN(dropout_rate=0.0).to(device)
history_overfit = train_model(cnn_overfit, train_loader_2d, X_test_2d, y_test_tensor, epochs=30)

In [ ]:
# 6.2 Entrenar CNN con Dropout
print('--- CNN con Dropout (p=0.3, 30 epochs) ---')
cnn_dropout = CNN(dropout_rate=0.3).to(device)
history_dropout = train_model(cnn_dropout, train_loader_2d, X_test_2d, y_test_tensor, epochs=30)

In [ ]:
# 6.3 Comparar curvas: overfitting visible
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Sin dropout
ax = axes[0]
ax.plot(history_overfit['train_loss'], color=ACCENT, lw=2, label='Train')
ax.plot(history_overfit['test_loss'], color=ACCENT2, lw=2, label='Test')
ax.set(xlabel='Epoch', ylabel='Loss', title='CNN sin Dropout — Loss')
ax.legend()
# Marcar zona de overfitting
gap = [t - v for t, v in zip(history_overfit['test_loss'], history_overfit['train_loss'])]
if max(gap) > 0.01:
    start_overfit = next((i for i, g in enumerate(gap) if g > 0.01), None)
    if start_overfit:
        ax.axvline(start_overfit, color='#ff4444', ls=':', lw=1.5, label=f'Overfitting ~epoch {start_overfit}')
        ax.legend()

# Con dropout
ax = axes[1]
ax.plot(history_dropout['train_loss'], color=ACCENT, lw=2, label='Train')
ax.plot(history_dropout['test_loss'], color=ACCENT2, lw=2, label='Test')
ax.set(xlabel='Epoch', ylabel='Loss', title='CNN con Dropout (0.3) — Loss')
ax.legend()

plt.suptitle('Efecto del Dropout en Overfitting', y=1.02, color='#fff', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

# Early stopping concepto
best_epoch_no_drop = np.argmin(history_overfit['test_loss'])
best_epoch_drop = np.argmin(history_dropout['test_loss'])
print(f'Sin dropout: mejor test loss en epoch {best_epoch_no_drop + 1}')
print(f'Con dropout: mejor test loss en epoch {best_epoch_drop + 1}')
print(f'\nEarly stopping = parar el entrenamiento en el mejor epoch de test loss.')

In [ ]:
# 6.4 Accuracy final CNN+Dropout
cnn_dropout.eval()
with torch.no_grad():
    y_pred_cnn_drop = cnn_dropout(X_test_2d).argmax(1).cpu().numpy()

acc_cnn_drop = accuracy_score(y_test, y_pred_cnn_drop)
n_params_cnn_drop = sum(p.numel() for p in cnn_dropout.parameters())

print(f'CNN sin Dropout: {acc_cnn:.4f}')
print(f'CNN con Dropout: {acc_cnn_drop:.4f}')

---
## 7. Comparacion final

> **Pregunta:** Cuanto mejora cada paso?

In [ ]:
# 7.1 Tabla comparativa
resumen = pd.DataFrame([
    {
        'Modelo': 'Random Forest',
        'Accuracy': f'{acc_rf:.4f}',
        'Tiempo (s)': f'{rf_train_time:.1f}',
        'Parametros': '~100 arboles',
    },
    {
        'Modelo': 'MLP (784->128->64->10)',
        'Accuracy': f'{acc_mlp:.4f}',
        'Tiempo (s)': f'{history_mlp["time"]:.1f}',
        'Parametros': f'{n_params_mlp:,}',
    },
    {
        'Modelo': 'CNN',
        'Accuracy': f'{acc_cnn:.4f}',
        'Tiempo (s)': f'{history_cnn["time"]:.1f}',
        'Parametros': f'{n_params_cnn:,}',
    },
    {
        'Modelo': 'CNN + Dropout (0.3)',
        'Accuracy': f'{acc_cnn_drop:.4f}',
        'Tiempo (s)': f'{history_dropout["time"]:.1f}',
        'Parametros': f'{n_params_cnn_drop:,}',
    },
])
print(resumen.to_markdown(index=False))

In [ ]:
# 7.2 Visualizacion: accuracy por modelo
models_names = ['Random\nForest', 'MLP', 'CNN', 'CNN+\nDropout']
accuracies = [acc_rf, acc_mlp, acc_cnn, acc_cnn_drop]
colors = [ACCENT2, ACCENT, ACCENT, ACCENT3]

fig, ax = plt.subplots(figsize=(10, 5))
bars = ax.bar(models_names, accuracies, color=colors, edgecolor='#333', alpha=0.85)
for bar, acc in zip(bars, accuracies):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.002,
            f'{acc:.3f}', ha='center', va='bottom', color='#e0e0e0', fontsize=12, fontweight='bold')
ax.set(ylabel='Accuracy', title='Comparacion Final — Accuracy en Test')
ax.set_ylim(min(accuracies) - 0.05, 1.0)
plt.tight_layout()
plt.show()

In [ ]:
# 7.3 Predicciones en las mismas imagenes de test — comparacion visual
np.random.seed(42)
sample_idxs = np.random.choice(len(y_test), 10, replace=False)

all_preds = {
    'Random Forest': y_pred_rf,
    'MLP': y_pred_mlp,
    'CNN': y_pred_cnn,
    'CNN+Dropout': y_pred_cnn_drop,
}

fig, axes = plt.subplots(len(all_preds) + 1, 10, figsize=(14, 8))

# Fila 0: imagenes
for i, idx in enumerate(sample_idxs):
    ax = axes[0, i]
    ax.imshow(X_test_raw[idx], cmap='gray')
    ax.set_title(f'Real: {y_test[idx]}', color=ACCENT2, fontsize=9)
    ax.set_xticks([])
    ax.set_yticks([])
axes[0, 0].set_ylabel('Imagen', color='#e0e0e0', fontsize=10)

# Filas 1-4: predicciones de cada modelo
for row, (name, preds) in enumerate(all_preds.items(), 1):
    for i, idx in enumerate(sample_idxs):
        ax = axes[row, i]
        pred = preds[idx]
        correct = pred == y_test[idx]
        color = ACCENT3 if correct else '#ff4444'
        ax.text(0.5, 0.5, str(pred), transform=ax.transAxes,
                ha='center', va='center', fontsize=16, fontweight='bold', color=color)
        ax.set_facecolor('#1a1a1a')
        ax.set_xticks([])
        ax.set_yticks([])
    axes[row, 0].set_ylabel(name, color='#e0e0e0', fontsize=9)

plt.suptitle('Predicciones por modelo en las mismas imagenes', y=1.02,
             color='#fff', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

---
## 8. Sintesis y conclusiones

### Cuando usar ML clasico vs deep learning?

| | ML clasico (RF, XGBoost) | Deep Learning (MLP, CNN) |
|---|---|---|
| **Datos** | Tablas, features manuales | Imagenes, texto, audio, secuencias |
| **Volumen** | Funciona con pocos datos | Necesita muchos datos |
| **Interpretabilidad** | Alta (feature importance, SHAP) | Baja (caja negra) |
| **Complejidad** | Baja, rapido de entrenar | Alta, necesita GPU para escalar |
| **Accuracy en imagenes** | Buena (~97%) | Mejor (~99%) |

### Trade-offs

- **Accuracy vs complejidad:** La CNN es mas precisa pero tiene mas parametros y tarda mas
- **Accuracy vs interpretabilidad:** Un Random Forest te dice que pixeles importan; una CNN es opaca
- **Regularizacion:** Dropout, early stopping y data augmentation son esenciales para evitar overfitting

### Siguientes pasos

- **Transfer learning:** Usar redes pre-entrenadas (ResNet, VGG) en vez de entrenar desde cero
- **Data augmentation:** Rotar, escalar, distorsionar imagenes para tener mas datos de entrenamiento
- **Arquitecturas modernas:** Transformers, attention mechanisms

In [ ]:
# 8.1 Resumen final
results = {
    'Random Forest': acc_rf,
    'MLP': acc_mlp,
    'CNN': acc_cnn,
    'CNN + Dropout': acc_cnn_drop,
}
best_name = max(results, key=results.get)
best_acc = results[best_name]

print('=' * 60)
print('  RESUMEN: Intro a Deep Learning con MNIST')
print('=' * 60)
print(f'\n  Random Forest:   {acc_rf:.4f}  (baseline ML clasico)')
print(f'  MLP:             {acc_mlp:.4f}  (red neuronal simple)')
print(f'  CNN:             {acc_cnn:.4f}  (red convolucional)')
print(f'  CNN + Dropout:   {acc_cnn_drop:.4f}  (con regularizacion)')
print(f'\n  Mejor modelo: {best_name}  ({best_acc:.4f})')
print(f'  Mejora sobre baseline: {(best_acc - acc_rf)*100:+.2f} puntos porcentuales')
print('=' * 60)